In [1]:
%cd ..
import pandas as pd
import re
from tqdm.auto import tqdm
from functools import partial

/home/kalisto/Documents/9_sem/BigDataAndML/Lossy-RAG


In [2]:
questions = pd.read_csv("data/questions.csv")
questions_paraphrased = pd.read_csv("data/questions_paraphrased.csv")
contexts = pd.read_csv("data/contexts.csv")
answers = pd.read_csv("data/answers.csv")

In [3]:
questions = questions[questions["role"] == "valid"]
questions_paraphrased = questions_paraphrased[questions_paraphrased["role"] == "valid"]

questions.reset_index()
questions_paraphrased.reset_index()
answers.reset_index()

answers = answers.groupby("orig_qid", as_index=False).agg(list)

pass

In [4]:
questions = questions[questions["orig_qid"].isin(questions_paraphrased["orig_qid"])]
questions = questions[~questions["is_paraphrase"]]

questions = pd.merge(questions, answers, on="orig_qid", how="left")
questions_paraphrased = pd.merge(questions_paraphrased, answers, on="orig_qid", how="left")

questions = pd.merge(questions, contexts, on="pid", how="left")
questions_paraphrased = pd.merge(questions_paraphrased, contexts, on="pid", how="left")

questions = questions.rename(columns={"text": "answer_text", "start": "answer_start_pos"})
questions_paraphrased = questions_paraphrased.rename(columns={"text": "answer_text", "start": "answer_start_pos"})

In [5]:
questions = questions.sort_values("orig_qid").reset_index(drop=True)
questions_paraphrased = questions_paraphrased.sort_values("orig_qid").reset_index(drop=True)

In [6]:
questions = questions.drop(columns=["squad_id"])
questions_paraphrased = questions_paraphrased.drop(columns=["squad_id"])

In [7]:
questions_paraphrased

,qid,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos,title,context,n_chunks
0,100417,88027,valid,True,18914,Where did the Panthers train ahead of the Supe...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]",Super_Bowl_50,The Panthers used the San Jose State practice ...,1
1,100463,88051,valid,True,18916,What hue were the balls given to many secondar...,"[gold, Gold]","[8, 357]",Super_Bowl_50,Various gold-themed promotions and initiatives...,1
2,100173,88112,valid,True,18920,Which program connects area firms with deals t...,[Business Connect],[164],Super_Bowl_50,"For the first time, the Super Bowl 50 Host Com...",1
3,100518,88115,valid,True,18921,What is the Host Committee's philanthropic pro...,"[the 50 fund, 50 fund]","[200, 204]",Super_Bowl_50,The Super Bowl 50 Host Committee has vowed to ...,1
4,100467,88144,valid,True,18923,With how many TV networks does the NFL have ai...,[three],[84],Super_Bowl_50,"In the United States, the game was televised b...",3
...,...,...,...,...,...,...,...,...,...,...,...
495,100330,98112,valid,True,20945,What is required to squeeze negatively charged...,"[energy, energy]","[579, 539]",Force,It is a common misconception to ascribe the st...,4
496,100287,98116,valid,True,20946,What interaction acts between protons and neut...,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]",Force,The strong force only acts directly upon eleme...,4
497,100582,98131,valid,True,20949,What ultimately results from attaching ever mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]",Force,Tension forces can be modeled using ideal stri...,6
498,100398,98133,valid,True,20950,To what objects did the physics of Newton apply?,"[idealized point particles, idealized point pa...","[100, 100]",Force,Newton's laws and Newtonian mechanics in gener...,3


In [8]:
questions

,qid,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos,title,context,n_chunks
0,88027,88027,valid,False,18914,Where was the practice place the Panthers used...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]",Super_Bowl_50,The Panthers used the San Jose State practice ...,1
1,88051,88051,valid,False,18916,What color were the footballs handed out to a ...,"[gold, Gold]","[8, 357]",Super_Bowl_50,Various gold-themed promotions and initiatives...,1
2,88112,88112,valid,False,18920,What is the name of the service that gets loca...,[Business Connect],[164],Super_Bowl_50,"For the first time, the Super Bowl 50 Host Com...",1
3,88115,88115,valid,False,18921,What is the name of the Host Committee's chari...,"[the 50 fund, 50 fund]","[200, 204]",Super_Bowl_50,The Super Bowl 50 Host Committee has vowed to ...,1
4,88144,88144,valid,False,18923,How many broadcast television partners does th...,[three],[84],Super_Bowl_50,"In the United States, the game was televised b...",3
...,...,...,...,...,...,...,...,...,...,...,...
495,98112,98112,valid,False,20945,What is needed to pack electrons densely toget...,"[energy, energy]","[579, 539]",Force,It is a common misconception to ascribe the st...,4
496,98116,98116,valid,False,20946,What is the force between nucleons?,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]",Force,The strong force only acts directly upon eleme...,4
497,98131,98131,valid,False,20949,What is the final effect of adding more and mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]",Force,Tension forces can be modeled using ideal stri...,6
498,98133,98133,valid,False,20950,What did Newton's mechanics affect?,"[idealized point particles, idealized point pa...","[100, 100]",Force,Newton's laws and Newtonian mechanics in gener...,3


In [9]:
import src.bench as bench
from src.utils.model_generate import opeanai_model_generate_single_answer
from IPython.display import display
from ipywidgets import Output

model_name = "Swift-1.5-Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS"
question_num = 1

gen = partial(
    opeanai_model_generate_single_answer,
    model_name=model_name,
    max_new_tokens=4096,
    reasoning_budget_tokens=3072,
    do_sample=True,
    temperature=1,
    top_p=0.95,
    top_k=20,
)

bars, summary, log = Output(), Output(), Output()
display(bars, summary, log)

with log:
    print("Log")

with bars:
    score, elapsed = bench.run_squad(gen, questions.head(question_num), responses_path=f"log/model_answers/squad_{model_name}.jsonl", log=log)
    score_p, elapsed_p = bench.run_squad(gen, questions_paraphrased.head(question_num), responses_path=f"log/model_answers/squad_p_{model_name}.jsonl", log=log)

with summary:
    print("─" * 80)
    print("Score without context")
    print("Num questions", question_num, "\n", "Score:", score, "\n", "Score paraphrase", score_p)
    print("─" * 80)

Output()

Output()

Output()

In [10]:
bars, summary, log = Output(), Output(), Output()
display(bars, summary, log)

with log:
    print("Log")

with bars:
    score, elapsed = bench.run_squad(gen, questions.head(question_num), use_context=True, responses_path=f"log/model_answers/squad_{model_name}.jsonl", log=log)
    score_p, elapsed_p = bench.run_squad(gen, questions_paraphrased.head(question_num), use_context=True, responses_path=f"log/model_answers/squad_p_{model_name}.jsonl", log=log)

with summary:
    print("─" * 80)
    print("Summary")
    print("Score with full context")
    print("Num questions", question_num, "\n", "Score:", score, "\n", "Score paraphrase", score_p)
    print("─" * 80)

Output()

Output()

Output()